# 4.1 Vectors in $\mathbb{R}^n$

**Course:** Elementary Linear Algebra (Larson, 8e)  
**Chapter 4:** Vector Spaces

---

### What you will learn

- How to represent vectors in $\mathbb{R}^n$ as **column matrices** (n-tuples).
- How **vector addition** and **scalar multiplication** work component-wise.
- What a **linear combination** $c_1\mathbf{v}_1 + c_2\mathbf{v}_2 + \cdots + c_k\mathbf{v}_k$ is and how to compute one.
- How the **standard unit vectors** $\mathbf{e}_1, \mathbf{e}_2, \ldots, \mathbf{e}_n$ form the building blocks of $\mathbb{R}^n$.
- How to determine whether a target vector can be expressed as a linear combination of given vectors by **solving a linear system**.
- What happens when the vectors are **linearly dependent** and some targets become unreachable.

---

## 1. Setup

In [ ]:
import sys, os
sys.path.insert(0, os.path.join(os.getcwd(), '../..'))

In [ ]:
from linalg_core.matrix import Matrix
from linalg_core.ops import add, scalar_mul, multiply
from linalg_core.elimination import solve
from linalg_core import EPSILON

import numpy as np
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D
import math

---

## 2. Motivation --- Robot Arm in 3D

A robot arm in 3D space has **three actuators**, each producing a force along a fixed
direction. The force vectors are:

$$\mathbf{f}_1 = \begin{bmatrix} 1 \\ 0 \\ 2 \end{bmatrix}, \quad
\mathbf{f}_2 = \begin{bmatrix} 0 \\ 1 \\ -1 \end{bmatrix}, \quad
\mathbf{f}_3 = \begin{bmatrix} 2 \\ 1 \\ 1 \end{bmatrix}$$

Each actuator can fire at any scalar intensity $c_i$. The total force produced is the
**linear combination**:

$$\mathbf{F}_{\text{total}} = c_1 \mathbf{f}_1 + c_2 \mathbf{f}_2 + c_3 \mathbf{f}_3$$

The robot must apply a target force:

$$\mathbf{t} = \begin{bmatrix} 4 \\ 2 \\ 6 \end{bmatrix}$$

**Question:** Can the robot reach this target? If so, what intensities $c_1, c_2, c_3$ are needed?

This is a fundamental question about vectors in $\mathbb{R}^n$:
can we write one vector as a linear combination of others?
We will build the tools to answer it in this notebook.

---

## 3. Build --- Core Concepts

We build up vectors in $\mathbb{R}^n$ from the ground up, starting with the
definition and progressing to linear combinations and the robot arm problem.

### 3.1 Vectors in $\mathbb{R}^n$ --- $n$-Tuples as Column Matrices

> **Definition (Larson, Sec. 4.1).** A **vector in $\mathbb{R}^n$** is an ordered
> $n$-tuple of real numbers:
>
> $$\mathbf{v} = \begin{bmatrix} v_1 \\ v_2 \\ \vdots \\ v_n \end{bmatrix} \in \mathbb{R}^n$$
>
> The number $v_i$ is called the $i$-th **component** (or **entry**) of $\mathbf{v}$.

In our library, we represent a vector in $\mathbb{R}^n$ as an $n \times 1$ **column matrix**
using `Matrix.from_vector`. This is not just a convention --- it lets us use matrix
operations (addition, scalar multiplication, matrix-vector products) directly on vectors.

**Two vectors are equal** if and only if they have the same number of components
and all corresponding components are equal:

$$\mathbf{u} = \mathbf{v} \quad \Longleftrightarrow \quad u_i = v_i \text{ for all } i = 1, 2, \ldots, n$$

The **zero vector** $\mathbf{0} \in \mathbb{R}^n$ has all components equal to zero.

In [ ]:
v1 = Matrix.from_vector([1, 0, 2])
v2 = Matrix.from_vector([0, 1, -1])
v3 = Matrix.from_vector([2, 1, 1])

print("Force vector f1 (a vector in R^3):")
print(v1)
print(f"\nDimensions: {v1.rows} x {v1.cols}  (column matrix)")
print(f"Component 1 (v[0,0]): {v1[0,0]}")
print(f"Component 2 (v[1,0]): {v1[1,0]}")
print(f"Component 3 (v[2,0]): {v1[2,0]}")

print("\nAll three force vectors:")
for label, v in [("f1", v1), ("f2", v2), ("f3", v3)]:
    print(f"  {label} = [{v[0,0]:.0f}, {v[1,0]:.0f}, {v[2,0]:.0f}]^T")

print("\nZero vector in R^3:")
zero = Matrix.from_vector([0, 0, 0])
print(zero)

### 3.2 Vector Addition and Scalar Multiplication

> **Definition (Larson, Sec. 4.1).** Let $\mathbf{u}, \mathbf{v} \in \mathbb{R}^n$
> and let $c$ be a scalar.
>
> **Vector addition:**
> $$\mathbf{u} + \mathbf{v} = \begin{bmatrix} u_1 + v_1 \\ u_2 + v_2 \\ \vdots \\ u_n + v_n \end{bmatrix}$$
>
> **Scalar multiplication:**
> $$c\mathbf{v} = \begin{bmatrix} cv_1 \\ cv_2 \\ \vdots \\ cv_n \end{bmatrix}$$

Both operations are performed **component-wise**. Since our vectors are stored as
$n \times 1$ column matrices, we can directly use `ops.add` and `ops.scalar_mul`
from our library --- no new implementation needed.

These two operations are the fundamental building blocks of $\mathbb{R}^n$.
Together, they define the **vector space structure** (which we formalize in Section 4.2).

In [ ]:
print("f1 ="); print(v1)
print("\nf2 ="); print(v2)

v_sum = add(v1, v2)
print("\nf1 + f2 ="); print(v_sum)
print("Component check:")
for i in range(3):
    print(f"  ({v1[i,0]:.0f}) + ({v2[i,0]:.0f}) = {v_sum[i,0]:.0f}")

print("\n--- Scalar Multiplication ---")
c = 2.5
v_scaled = scalar_mul(v1, c)
print(f"\n{c} * f1 ="); print(v_scaled)
print("Component check:")
for i in range(3):
    print(f"  {c} * ({v1[i,0]:.0f}) = {v_scaled[i,0]:.1f}")

### 3.3 Linear Combinations

> **Definition (Larson, Sec. 4.1).** A vector $\mathbf{w}$ is a **linear combination**
> of vectors $\mathbf{v}_1, \mathbf{v}_2, \ldots, \mathbf{v}_k$ if there exist
> scalars $c_1, c_2, \ldots, c_k$ such that
>
> $$\mathbf{w} = c_1 \mathbf{v}_1 + c_2 \mathbf{v}_2 + \cdots + c_k \mathbf{v}_k$$
>
> The scalars $c_1, \ldots, c_k$ are called the **coefficients** of the linear combination.

Linear combinations are the central concept of this section. They tell us
how to **build new vectors from old ones** using only addition and scalar multiplication.

We implement a general `linear_combination` function that takes a list of column-matrix
vectors and a list of scalar coefficients.

In [ ]:
def linear_combination(vectors, scalars):
    """Compute c1*v1 + c2*v2 + ... + ck*vk.

    Parameters
    ----------
    vectors : list of Matrix
        Column matrices (n x 1), all the same dimension.
    scalars : list of float
        Coefficients c1, c2, ..., ck.

    Returns
    -------
    Matrix
        The n x 1 column matrix representing the linear combination.
    """
    if len(vectors) != len(scalars):
        raise ValueError(f"Need same number of vectors ({len(vectors)}) and scalars ({len(scalars)})")
    if not vectors:
        raise ValueError("Need at least one vector")

    n = vectors[0].rows
    result = Matrix(n, 1)
    for v, c in zip(vectors, scalars):
        result = add(result, scalar_mul(v, c))
    return result


combo = linear_combination([v1, v2, v3], [1, 2, -1])
print("Linear combination: 1*f1 + 2*f2 + (-1)*f3")
print()
print("Step by step:")
print(f"  1 * f1 = 1 * [1, 0, 2]^T  = [1, 0, 2]^T")
print(f"  2 * f2 = 2 * [0, 1, -1]^T = [0, 2, -2]^T")
print(f" -1 * f3 = -1 * [2, 1, 1]^T = [-2, -1, -1]^T")
print(f"  Sum:                        = [{combo[0,0]:.0f}, {combo[1,0]:.0f}, {combo[2,0]:.0f}]^T")
print()
print("Result:"); print(combo)

### 3.4 Standard Unit Vectors

> **Definition (Larson, Sec. 4.1).** The **standard unit vectors** in $\mathbb{R}^n$ are
> the $n$ vectors $\mathbf{e}_1, \mathbf{e}_2, \ldots, \mathbf{e}_n$ defined by
>
> $$\mathbf{e}_i = \begin{bmatrix} 0 \\ \vdots \\ 1 \\ \vdots \\ 0 \end{bmatrix} \leftarrow i\text{-th position}$$
>
> That is, $\mathbf{e}_i$ has a $1$ in position $i$ and $0$s everywhere else.

**Key fact:** Every vector in $\mathbb{R}^n$ is a linear combination of the standard unit vectors:

$$\mathbf{v} = \begin{bmatrix} v_1 \\ v_2 \\ \vdots \\ v_n \end{bmatrix}
= v_1 \mathbf{e}_1 + v_2 \mathbf{e}_2 + \cdots + v_n \mathbf{e}_n$$

This means the standard unit vectors "generate" all of $\mathbb{R}^n$.
We will revisit this idea when we study **spanning sets** and **bases** later in Chapter 4.

Note that the standard unit vectors are precisely the **columns of the identity matrix** $I_n$.

In [ ]:
def standard_basis(n):
    """Return the standard basis {e1, e2, ..., en} for R^n as column matrices."""
    basis = []
    for i in range(n):
        data = [0.0] * n
        data[i] = 1.0
        basis.append(Matrix.from_vector(data))
    return basis


e1, e2, e3 = standard_basis(3)
print("Standard unit vectors in R^3:")
print("e1 ="); print(e1)
print("\ne2 ="); print(e2)
print("\ne3 ="); print(e3)

print("\n--- Connection to the identity matrix ---")
I3 = Matrix.identity(3)
print("I_3 ="); print(I3)
print(f"\nColumn 0 of I_3: {I3.get_col(0)}  (= e1)")
print(f"Column 1 of I_3: {I3.get_col(1)}  (= e2)")
print(f"Column 2 of I_3: {I3.get_col(2)}  (= e3)")

print("\n--- Every vector is a linear combination of standard basis ---")
print("\nf1 = [1, 0, 2]^T")
print("    = 1*e1 + 0*e2 + 2*e3")

reconstruction = linear_combination([e1, e2, e3], [1, 0, 2])
print(f"\nReconstructed: [{reconstruction[0,0]:.0f}, {reconstruction[1,0]:.0f}, {reconstruction[2,0]:.0f}]^T")
print(f"Matches f1? {reconstruction == v1}")

print("\n--- R^n notation ---")
for n in [2, 3, 4]:
    basis = standard_basis(n)
    labels = [f"e{i+1}" for i in range(n)]
    print(f"R^{n} has {n} standard unit vectors: {', '.join(labels)}")

### 3.5 Robot Arm Problem --- Solving for Coefficients

We return to our motivating problem. We need to find scalars $c_1, c_2, c_3$ such that

$$c_1 \begin{bmatrix} 1 \\ 0 \\ 2 \end{bmatrix}
+ c_2 \begin{bmatrix} 0 \\ 1 \\ -1 \end{bmatrix}
+ c_3 \begin{bmatrix} 2 \\ 1 \\ 1 \end{bmatrix}
= \begin{bmatrix} 4 \\ 2 \\ 6 \end{bmatrix}$$

Writing this component-wise gives the linear system:

$$\begin{cases}
1\cdot c_1 + 0\cdot c_2 + 2\cdot c_3 = 4 \\
0\cdot c_1 + 1\cdot c_2 + 1\cdot c_3 = 2 \\
2\cdot c_1 + (-1)\cdot c_2 + 1\cdot c_3 = 6
\end{cases}$$

This is exactly the system $A\mathbf{c} = \mathbf{t}$ where the **columns of $A$**
are the force vectors $\mathbf{f}_1, \mathbf{f}_2, \mathbf{f}_3$.

We form the augmented matrix $[\mathbf{f}_1 \mid \mathbf{f}_2 \mid \mathbf{f}_3 \mid \mathbf{t}]$
and solve using RREF.

In [ ]:
f1 = [1, 0, 2]
f2 = [0, 1, -1]
f3 = [2, 1, 1]
t  = [4, 2, 6]

aug = Matrix.from_lists([
    [f1[0], f2[0], f3[0], t[0]],
    [f1[1], f2[1], f3[1], t[1]],
    [f1[2], f2[2], f3[2], t[2]],
])

print("Augmented matrix [f1 | f2 | f3 | t]:")
print(aug)

sol_type, result = solve(aug, verbose=True)

print(f"\nSolution type: {sol_type}")

if sol_type == "unique":
    c1, c2, c3 = result
    print(f"c1 = {c1:.4f}, c2 = {c2:.4f}, c3 = {c3:.4f}")
    print(f"\nThe robot reaches the target with intensities:")
    print(f"  Actuator 1: c1 = {c1:.1f}")
    print(f"  Actuator 2: c2 = {c2:.1f}")
    print(f"  Actuator 3: c3 = {c3:.1f}")

In [ ]:
print("--- Verification ---")
print()

vecs = [Matrix.from_vector(f1), Matrix.from_vector(f2), Matrix.from_vector(f3)]
target = Matrix.from_vector(t)
achieved = linear_combination(vecs, result)

print("Computed force:")
for c, f, label in zip(result, [f1, f2, f3], ['f1', 'f2', 'f3']):
    scaled = [c * x for x in f]
    print(f"  {c:+.4f} * {label} = [{scaled[0]:+.4f}, {scaled[1]:+.4f}, {scaled[2]:+.4f}]^T")

print(f"\nTotal force:  [{achieved[0,0]:.4f}, {achieved[1,0]:.4f}, {achieved[2,0]:.4f}]^T")
print(f"Target:       [{target[0,0]:.4f}, {target[1,0]:.4f}, {target[2,0]:.4f}]^T")
print(f"Match? {achieved == target}")

### 3.6 When It Fails --- Linear Dependence

What happens if one of our force vectors is a linear combination of the others?
Let's replace $\mathbf{f}_3 = [2, 1, 1]^T$ with

$$\mathbf{f}_3' = \begin{bmatrix} 3 \\ 1 \\ 5 \end{bmatrix}$$

Notice that $\mathbf{f}_3' = 3\mathbf{f}_1 + \mathbf{f}_2$:

$$3 \begin{bmatrix} 1 \\ 0 \\ 2 \end{bmatrix} + 1 \begin{bmatrix} 0 \\ 1 \\ -1 \end{bmatrix}
= \begin{bmatrix} 3 \\ 1 \\ 5 \end{bmatrix}$$

The new $\mathbf{f}_3'$ contributes **no new direction** --- it is a redundant force.
The three vectors are **linearly dependent**: they span only a 2D plane inside $\mathbb{R}^3$.

Some targets that lie off this plane become **unreachable**.
This is a preview of **linear independence** (Section 4.4) and **span** (Section 4.5):
three vectors in $\mathbb{R}^3$ span all of $\mathbb{R}^3$ *only if* they are linearly independent.

In [ ]:
f3_dep = [3, 1, 5]

print("Checking the dependence relation:")
print(f"3 * f1 + 1 * f2 = 3*{f1} + 1*{f2}")
check = [3*f1[i] + 1*f2[i] for i in range(3)]
print(f"             = {check}")
print(f"        f3'  = {f3_dep}")
print(f"Match: {check == f3_dep}")
print()
print("Since f3' = 3*f1 + f2, the vectors {f1, f2, f3'} are LINEARLY DEPENDENT.")
print("They span only a 2D plane in R^3, not all of R^3.")

In [ ]:
print("--- Can we still reach t = [4, 2, 6]^T? ---")
print()

aug_dep = Matrix.from_lists([
    [f1[0], f2[0], f3_dep[0], t[0]],
    [f1[1], f2[1], f3_dep[1], t[1]],
    [f1[2], f2[2], f3_dep[2], t[2]],
])

sol_dep, res_dep = solve(aug_dep)
print(f"Solution type: {sol_dep}")

if sol_dep == "infinite":
    print(f"\nYes, but with infinitely many solutions (the system is underdetermined).")
    print(f"Free variables: {res_dep['free_vars']}")
    print(f"Particular solution: {res_dep['particular']}")
    print(f"Direction: {res_dep['directions']}")
    print(f"\nThe target [4,2,6]^T happens to lie in the plane spanned by f1 and f2.")

print()
print("--- Now try a target OFF the plane: t' = [1, 0, 0]^T ---")
print()

t_bad = [1, 0, 0]
aug_bad = Matrix.from_lists([
    [f1[0], f2[0], f3_dep[0], t_bad[0]],
    [f1[1], f2[1], f3_dep[1], t_bad[1]],
    [f1[2], f2[2], f3_dep[2], t_bad[2]],
])

sol_bad, res_bad = solve(aug_bad)
print(f"Solution type: {sol_bad}")

if sol_bad == "inconsistent":
    print(f"\nThe target t' = [1, 0, 0]^T is UNREACHABLE!")
    print("With linearly dependent forces, the robot cannot produce")
    print("forces in every direction of R^3.")
    print("\nLesson: three vectors in R^3 that are linearly dependent")
    print("span at most a 2D subspace. Some targets are forever out of reach.")

---

## 4. Verify --- Cross-Check with NumPy

We verify our from-scratch results against NumPy to build confidence.

In [ ]:
def to_np(mat):
    """Convert our Matrix to a NumPy array."""
    return np.array(mat.to_lists())

def check_match(ours, theirs, label):
    """Compare our Matrix result against a NumPy array."""
    ours_np = to_np(ours)
    match = np.allclose(ours_np, theirs, atol=1e-9)
    status = "PASS" if match else "FAIL"
    print(f"[{status}] {label}")
    if not match:
        print(f"  Ours:   {ours_np.flatten()}")
        print(f"  NumPy:  {theirs.flatten()}")
    return match

In [ ]:
f1_np = np.array([[1], [0], [2]], dtype=float)
f2_np = np.array([[0], [1], [-1]], dtype=float)
f3_np = np.array([[2], [1], [1]], dtype=float)
t_np  = np.array([[4], [2], [6]], dtype=float)

check_match(
    add(Matrix.from_vector([1,0,2]), Matrix.from_vector([0,1,-1])),
    f1_np + f2_np,
    "Vector addition: f1 + f2"
)

check_match(
    scalar_mul(Matrix.from_vector([1,0,2]), 2.5),
    2.5 * f1_np,
    "Scalar multiplication: 2.5 * f1"
)

combo_np = 1 * f1_np + 2 * f2_np + (-1) * f3_np
check_match(
    linear_combination(
        [Matrix.from_vector([1,0,2]), Matrix.from_vector([0,1,-1]), Matrix.from_vector([2,1,1])],
        [1, 2, -1]
    ),
    combo_np,
    "Linear combination: 1*f1 + 2*f2 - 1*f3"
)

print()
print("--- Robot arm system verification ---")
A_np = np.column_stack([f1_np, f2_np, f3_np])
c_np = np.linalg.solve(A_np, t_np)
print(f"NumPy solution: c = {c_np.flatten()}")

residual = A_np @ c_np - t_np
match = np.allclose(residual, 0, atol=1e-9)
print(f"[{'PASS' if match else 'FAIL'}] NumPy verification: A @ c == t")

our_c = np.array(result)
coeff_match = np.allclose(our_c, c_np.flatten(), atol=1e-6)
print(f"[{'PASS' if coeff_match else 'FAIL'}] Our coefficients match NumPy's")

print()
print("--- Dependence check ---")
f3_dep_np = np.array([[3], [1], [5]], dtype=float)
A_dep_np = np.column_stack([f1_np, f2_np, f3_dep_np])
rank_dep = np.linalg.matrix_rank(A_dep_np)
print(f"Rank of [f1|f2|f3'] = {rank_dep}  (< 3, confirming dependence)")
print(f"[{'PASS' if rank_dep < 3 else 'FAIL'}] f1, f2, f3' are linearly dependent")

rank_ind = np.linalg.matrix_rank(A_np)
print(f"\nRank of [f1|f2|f3]  = {rank_ind}  (= 3, confirming independence)")
print(f"[{'PASS' if rank_ind == 3 else 'FAIL'}] f1, f2, f3 are linearly independent")

---

## 5. Visualize

### 5.1 2D Parallelogram Rule for Linear Combinations

In $\mathbb{R}^2$, the sum of two vectors $\mathbf{u} + \mathbf{v}$ can be visualized
as the diagonal of the **parallelogram** formed by $\mathbf{u}$ and $\mathbf{v}$.

More generally, a linear combination $c_1 \mathbf{u} + c_2 \mathbf{v}$ first scales
each vector, then sums them using the parallelogram rule.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

u = np.array([2, 1])
v = np.array([1, 3])

ax = axes[0]
ax.set_title('Vector Addition: u + v', fontsize=11, fontweight='bold')
ax.annotate('', xy=u, xytext=(0,0),
            arrowprops=dict(arrowstyle='->', color='steelblue', lw=2.2))
ax.annotate('', xy=v, xytext=(0,0),
            arrowprops=dict(arrowstyle='->', color='tomato', lw=2.2))
ax.annotate('', xy=u+v, xytext=(0,0),
            arrowprops=dict(arrowstyle='->', color='green', lw=2.5))

parallelogram = np.array([[0,0], u, u+v, v, [0,0]])
ax.fill(parallelogram[:,0], parallelogram[:,1], alpha=0.1, color='green')
ax.plot(parallelogram[:,0], parallelogram[:,1], '--', color='gray', alpha=0.5, lw=1)

ax.text(u[0]*0.5-0.3, u[1]*0.5+0.2, r'$\mathbf{u}$', fontsize=13,
        color='steelblue', fontweight='bold')
ax.text(v[0]*0.5+0.15, v[1]*0.5-0.1, r'$\mathbf{v}$', fontsize=13,
        color='tomato', fontweight='bold')
ax.text((u[0]+v[0])*0.5+0.15, (u[1]+v[1])*0.5+0.15, r'$\mathbf{u+v}$',
        fontsize=13, color='green', fontweight='bold')

c1_vis, c2_vis = 1.5, 0.8
ax = axes[1]
ax.set_title(f'Linear Combination: {c1_vis}u + {c2_vis}v', fontsize=11, fontweight='bold')
cu = c1_vis * u
cv = c2_vis * v
lc_result = cu + cv

ax.annotate('', xy=u, xytext=(0,0),
            arrowprops=dict(arrowstyle='->', color='steelblue', lw=1.2, linestyle='--'))
ax.annotate('', xy=v, xytext=(0,0),
            arrowprops=dict(arrowstyle='->', color='tomato', lw=1.2, linestyle='--'))

ax.annotate('', xy=cu, xytext=(0,0),
            arrowprops=dict(arrowstyle='->', color='steelblue', lw=2.2))
ax.annotate('', xy=cv, xytext=(0,0),
            arrowprops=dict(arrowstyle='->', color='tomato', lw=2.2))
ax.annotate('', xy=lc_result, xytext=(0,0),
            arrowprops=dict(arrowstyle='->', color='green', lw=2.5))

para2 = np.array([[0,0], cu, lc_result, cv, [0,0]])
ax.fill(para2[:,0], para2[:,1], alpha=0.1, color='green')
ax.plot(para2[:,0], para2[:,1], '--', color='gray', alpha=0.5, lw=1)

ax.text(cu[0]*0.5-0.4, cu[1]*0.5+0.2, f'{c1_vis}u', fontsize=12,
        color='steelblue', fontweight='bold')
ax.text(cv[0]*0.5+0.15, cv[1]*0.5-0.1, f'{c2_vis}v', fontsize=12,
        color='tomato', fontweight='bold')
ax.text(lc_result[0]*0.5+0.15, lc_result[1]*0.5+0.2,
        f'{c1_vis}u+{c2_vis}v', fontsize=11, color='green', fontweight='bold')

ax = axes[2]
ax.set_title('Standard Basis Decomposition', fontsize=11, fontweight='bold')
w = np.array([3, 2])

ax.annotate('', xy=w, xytext=(0,0),
            arrowprops=dict(arrowstyle='->', color='green', lw=2.5))
ax.annotate('', xy=[w[0], 0], xytext=(0,0),
            arrowprops=dict(arrowstyle='->', color='steelblue', lw=2))
ax.annotate('', xy=[0, w[1]], xytext=(0,0),
            arrowprops=dict(arrowstyle='->', color='tomato', lw=2))

ax.plot([w[0], w[0]], [0, w[1]], '--', color='gray', alpha=0.5, lw=1)
ax.plot([0, w[0]], [w[1], w[1]], '--', color='gray', alpha=0.5, lw=1)

ax.text(w[0]*0.5+0.1, -0.35, r'$3\mathbf{e}_1$', fontsize=12,
        color='steelblue', fontweight='bold')
ax.text(-0.6, w[1]*0.5, r'$2\mathbf{e}_2$', fontsize=12,
        color='tomato', fontweight='bold')
ax.text(w[0]+0.15, w[1]+0.1, r'$\mathbf{w}=[3,2]^T$', fontsize=11,
        color='green', fontweight='bold')

for ax in axes:
    ax.set_xlim(-0.8, 5)
    ax.set_ylim(-0.8, 5)
    ax.set_aspect('equal')
    ax.grid(True, alpha=0.3)
    ax.axhline(0, color='gray', linewidth=0.5)
    ax.axvline(0, color='gray', linewidth=0.5)
    ax.plot(0, 0, 'ko', markersize=4)

fig.suptitle('2D Vector Operations and the Parallelogram Rule',
             fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

### 5.2 3D Force Vectors Reaching the Target

We visualize the robot arm problem in 3D: each force vector is drawn from the origin,
and the scaled forces are chained head-to-tail to reach the target.

In [ ]:
fig = plt.figure(figsize=(15, 6))

f1_arr = np.array([1, 0, 2], dtype=float)
f2_arr = np.array([0, 1, -1], dtype=float)
f3_arr = np.array([2, 1, 1], dtype=float)
t_arr  = np.array([4, 2, 6], dtype=float)

coeffs = result

ax1 = fig.add_subplot(121, projection='3d')
ax1.set_title('Force Vectors (unscaled)', fontsize=11, fontweight='bold')

origin = [0, 0, 0]
colors = ['steelblue', 'tomato', '#2ca02c']
labels = [r'$\mathbf{f}_1$', r'$\mathbf{f}_2$', r'$\mathbf{f}_3$']
forces = [f1_arr, f2_arr, f3_arr]

for f, color, label in zip(forces, colors, labels):
    ax1.quiver(*origin, *f, color=color, arrow_length_ratio=0.12,
               linewidth=2.5, label=label)

ax1.quiver(*origin, *t_arr, color='gold', arrow_length_ratio=0.1,
           linewidth=3, label=r'target $\mathbf{t}$')
ax1.scatter(*t_arr, color='gold', s=80, zorder=5, edgecolor='black')

ax1.set_xlabel('X'); ax1.set_ylabel('Y'); ax1.set_zlabel('Z')
ax1.legend(fontsize=9, loc='upper left')

ax2 = fig.add_subplot(122, projection='3d')
ax2.set_title('Scaled Forces Reaching Target', fontsize=11, fontweight='bold')

cumulative = np.array([0.0, 0.0, 0.0])
scaled_labels = [
    f'$c_1\\mathbf{{f}}_1$ ({coeffs[0]:.1f})',
    f'$c_2\\mathbf{{f}}_2$ ({coeffs[1]:.1f})',
    f'$c_3\\mathbf{{f}}_3$ ({coeffs[2]:.1f})',
]

for i, (f, c, color, label) in enumerate(zip(forces, coeffs, colors, scaled_labels)):
    scaled = c * f
    ax2.quiver(*cumulative, *scaled, color=color, arrow_length_ratio=0.1,
               linewidth=2.5, label=label)
    cumulative = cumulative + scaled

ax2.scatter(*t_arr, color='gold', s=100, zorder=5, edgecolor='black',
            label=r'target $\mathbf{t}$')
ax2.scatter(*cumulative, color='red', s=60, zorder=5, marker='x', linewidths=2)

ax2.set_xlabel('X'); ax2.set_ylabel('Y'); ax2.set_zlabel('Z')
ax2.legend(fontsize=8, loc='upper left')

for ax in [ax1, ax2]:
    all_pts = np.array([origin, f1_arr, f2_arr, f3_arr, t_arr])
    pad = 1.5
    ax.set_xlim(all_pts[:,0].min() - pad, all_pts[:,0].max() + pad)
    ax.set_ylim(all_pts[:,1].min() - pad, all_pts[:,1].max() + pad)
    ax.set_zlim(all_pts[:,2].min() - pad, all_pts[:,2].max() + pad)

fig.suptitle('Robot Arm: 3D Force Vectors and Linear Combination',
             fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

---

## 6. Exercises

Test your understanding with the following problems. Write your solutions
in the code cells provided.

### Exercise 1 (Easy)

Express the vector $\mathbf{v} = \begin{bmatrix} 3 \\ 7 \end{bmatrix}$
as a linear combination of the standard unit vectors
$\mathbf{e}_1 = \begin{bmatrix} 1 \\ 0 \end{bmatrix}$ and
$\mathbf{e}_2 = \begin{bmatrix} 0 \\ 1 \end{bmatrix}$.

Use `linear_combination` to verify: $\mathbf{v} = c_1 \mathbf{e}_1 + c_2 \mathbf{e}_2$.
What are $c_1$ and $c_2$?

In [ ]:
# YOUR CODE HERE


### Exercise 2 (Medium)

Determine whether $\mathbf{w} = \begin{bmatrix} 1 \\ 2 \\ 3 \end{bmatrix}$
can be expressed as a linear combination of
$\mathbf{v}_1 = \begin{bmatrix} 1 \\ 0 \\ 1 \end{bmatrix}$ and
$\mathbf{v}_2 = \begin{bmatrix} 0 \\ 1 \\ 1 \end{bmatrix}$.

Set up the augmented matrix $[\mathbf{v}_1 \mid \mathbf{v}_2 \mid \mathbf{w}]$
and use `solve` to find coefficients, or determine that no solution exists.

If a solution exists, verify it using `linear_combination`.

In [ ]:
# YOUR CODE HERE


### Exercise 3 (Challenge)

Write a function `is_linear_combination(target, vectors)` that takes:
- `target`: a list of floats representing a vector in $\mathbb{R}^n$
- `vectors`: a list of lists, each representing a vector in $\mathbb{R}^n$

and returns a tuple `(is_combo, coefficients)` where:
- `is_combo` is `True` if the target is a linear combination of the vectors, `False` otherwise
- `coefficients` is a list of floats if `is_combo` is `True`, or `None` otherwise

Test your function on:
1. `target = [4, 2, 6]`, `vectors = [[1,0,2], [0,1,-1], [2,1,1]]` (should succeed --- our robot arm)
2. `target = [1, 1, 1]`, `vectors = [[1,0,0], [0,1,0]]` (should fail --- can't reach $z=1$)
3. `target = [0, 0, 0]`, `vectors = [[1,2,3], [4,5,6]]` (should succeed --- the zero vector is always reachable)

In [ ]:
# YOUR CODE HERE


---

## Summary

| Concept | Key Takeaway |
|---|---|
| **Vectors in $\mathbb{R}^n$** | Ordered $n$-tuples; we represent them as $n \times 1$ column matrices |
| **Vector addition** | Component-wise: $(\mathbf{u}+\mathbf{v})_i = u_i + v_i$ |
| **Scalar multiplication** | Component-wise: $(c\mathbf{v})_i = c \cdot v_i$ |
| **Linear combination** | $c_1\mathbf{v}_1 + \cdots + c_k\mathbf{v}_k$; the fundamental way to build vectors from vectors |
| **Standard unit vectors** | $\mathbf{e}_1, \ldots, \mathbf{e}_n$ generate all of $\mathbb{R}^n$: every $\mathbf{v} = v_1\mathbf{e}_1 + \cdots + v_n\mathbf{e}_n$ |
| **Finding coefficients** | "Is $\mathbf{t}$ a combination of $\mathbf{v}_1, \ldots, \mathbf{v}_k$?" reduces to solving $[\mathbf{v}_1 \mid \cdots \mid \mathbf{v}_k \mid \mathbf{t}]$ |
| **Linear dependence** | When vectors are dependent, not every target is reachable --- the span is a proper subspace |